In [ ]:
import pandas as pd
import numpy as np
import math
import random
from typing import List, Dict, Tuple

# Wilaya coordinates in Algeria
town_coords = {
    'Adrar': (27.8749, -0.2930), 'Chlef': (36.1650, 1.3347),
    'Laghouat': (33.8078, 2.8808), 'Oum El Bouaghi': (35.8752, 7.1136),
    'Batna': (35.5559, 6.1741), 'Béjaïa': (36.7509, 5.0567),
    'Biskra': (34.8603, 5.7288), 'Béchar': (31.6237, -2.2164),
    'Blida': (36.4706, 2.8287), 'Bouira': (36.3763, 3.9000),
    'Tamanrasset': (22.7903, 5.5193), 'Tébessa': (35.4072, 8.1200),
    'Tlemcen': (34.8828, -1.3167), 'Tiaret': (35.3700, 1.3200),
    'Tizi Ouzou': (36.7167, 4.0500), 'Algiers': (36.7372, 3.0872),
    'Djelfa': (34.6667, 3.2500), 'Jijel': (36.8206, 5.7667),
    'Sétif': (36.1914, 5.4136), 'Saïda': (34.8414, 0.1514),
    'Skikda': (36.8792, 6.9067), 'Sidi Bel Abbès': (35.1939, -0.6414),
    'Annaba': (36.9000, 7.7667), 'Guelma': (36.4667, 7.4333),
    'Constantine': (36.3650, 6.6147), 'Médéa': (36.2675, 2.7500),
    'Mostaganem': (35.9333, 0.0833), 'M\'Sila': (35.7058, 4.5419),
    'Mascara': (35.3983, 0.1467), 'Ouargla': (31.9500, 5.3167),
    'Oran': (35.6911, -0.6417), 'El Bayadh': (33.6833, 1.0167),
    'Illizi': (26.4833, 8.4667), 'Bordj Bou Arréridj': (36.0667, 4.7667),
    'Boumerdès': (36.7667, 3.4667), 'El Tarf': (36.7667, 8.3167),
    'Tindouf': (27.6742, -8.1478), 'Tissemsilt': (35.6072, 1.8106),
    'El Oued': (33.3683, 6.8672), 'Khenchela': (35.4167, 7.1333),
    'Souk Ahras': (36.2833, 7.9500), 'Tipaza': (36.5897, 2.4475),
    'Mila': (36.4503, 6.2644), 'Aïn Defla': (36.2642, 1.9678),
    'Naama': (33.2667, -0.3167), 'Aïn Témouchent': (35.3028, -1.1414),
    'Ghardaïa': (32.4833, 3.6667), 'Relizane': (35.7372, 0.5558),
    'El M\'Ghair': (33.9500, 5.9167), 'El Menia': (30.5833, 2.8833),
    'Ouled Djellal': (34.4333, 5.0667), 'Bordj Badji Mokhtar': (21.3167, 0.9500),
    'Béni Abbès': (30.1333, -2.1667), 'Timimoun': (29.2500, 0.2333),
    'Touggourt': (33.1000, 6.0667), 'Djanet': (24.5553, 9.4892),
    'In Salah': (27.2167, 2.4667), 'In Guezzam': (19.8500, 5.7333)
}

def haversine(coord1: Tuple[float, float], coord2: Tuple[float, float]) -> float:
    lat1, lon1 = np.radians(coord1)
    lat2, lon2 = np.radians(coord2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371 * c

def load_data() -> pd.DataFrame:
    try:
        seekers_df = pd.read_csv('emplo.csv')
        if 'employee_Id' not in seekers_df.columns:
            seekers_df['employee_Id'] = [f'EMP-{i+1:03}' for i in range(len(seekers_df))]
    except FileNotFoundError:
        seekers_df = pd.DataFrame([{
            'employee_Id': 'EMP-001', 'gender': 'Male', 'birth_date': '1990-05-15',
            'age': 33, 'city': 'Algiers', 'wilaya': 'Algiers', 'department': 'IT',
            'sector': 'Technology', 'years_experience': 5, 'highest_education': 'Master',
            'contract_type': 'Full-time', 'technical_skills': 'Python, SQL, Spark',
            'language_proficiency': 'English, French', 'education_history': 'University of Algiers',
            'edu_value': 3, 'salary': 70000
        }])
    if 'wilaya' not in seekers_df.columns:
        seekers_df['wilaya'] = seekers_df.get('city', 'Algiers')
    seekers_df['wilaya'] = seekers_df['wilaya'].str.title()
    invalid = seekers_df[~seekers_df['wilaya'].isin(town_coords)]
    if not invalid.empty:
        print(f"Warning: Invalid wilayas: {invalid['wilaya'].unique()}")
    return seekers_df

def input_job_details(seekers_df: pd.DataFrame) -> pd.DataFrame:
    print("\n=== Enter Job Details ===")
    job_id = input("Job ID [JOB-001]: ").strip() or "JOB-001"
    title = input("Title [Data Engineer]: ").strip() or "Data Engineer"
    req_skills = input("Required Skills [Python, SQL, Spark]: ").strip()
    required_skills = [s.strip() for s in req_skills.split(',')] if req_skills else ['Python', 'SQL', 'Spark']
    min_exp = int(input("Min Experience [3]: ").strip() or 3)
    min_edu = input("Min Education [Master]: ").strip() or "Master"
    salary = float(input("Salary [75000]: ").strip() or 75000)
    location = ""
    while location not in town_coords:
        location = input("Wilaya [Algiers]: ").strip().title() or "Algiers"
        if location not in town_coords:
            print("Invalid wilaya. Try: Algiers, Oran, Constantine, etc.")
    sector = input("Sector [Technology]: ").strip() or "Technology"
    contract = input("Contract [Full-time]: ").strip() or "Full-time"
    return pd.DataFrame([{
        'job_id': job_id, 'title': title, 'required_skills': required_skills,
        'min_experience': min_exp, 'min_education': min_edu, 'salary_offer': salary,
        'location': location, 'sector': sector, 'contract_type': contract
    }])

def preprocess_data(seekers_df, jobs_df):
    edu_map = seekers_df.groupby('highest_education')['edu_value'].first().to_dict()
    seekers_df['edu_rank'] = seekers_df['edu_value']
    jobs_df['edu_rank'] = jobs_df['min_education'].map(edu_map).fillna(-1).astype(int)
    seekers_df['technical_skills'] = seekers_df['technical_skills'].fillna('').str.lower()
    seeker_skills = [set(s.split(', ')) for s in seekers_df['technical_skills']]
    job_skills = [set(map(str.lower, req)) for req in jobs_df['required_skills']]
    all_sectors = pd.Categorical(seekers_df['sector'].tolist() + jobs_df['sector'].tolist())
    seek_sector = pd.Categorical(seekers_df['sector'], categories=all_sectors.categories).codes
    job_sector = pd.Categorical(jobs_df['sector'], categories=all_sectors.categories).codes
    all_contracts = pd.Categorical(seekers_df['contract_type'].tolist() + jobs_df['contract_type'].tolist())
    seek_contract = pd.Categorical(seekers_df['contract_type'], categories=all_contracts.categories).codes
    job_contract = pd.Categorical(jobs_df['contract_type'], categories=all_contracts.categories).codes
    seekers_coords = seekers_df['wilaya'].map(lambda w: town_coords.get(w, (np.nan, np.nan))).tolist()
    jobs_coords = jobs_df['location'].map(lambda w: town_coords.get(w, (np.nan, np.nan))).tolist()
    return (seekers_df, jobs_df, seeker_skills, job_skills, seek_sector, job_sector,
            seek_contract, job_contract, seekers_coords, jobs_coords)

def calculate_features(seekers_df, jobs_df, seeker_skills, job_skills, seek_sector, job_sector,
                       seek_contract, job_contract, seekers_coords, jobs_coords) -> np.ndarray:
    N_seek, N_job = len(seekers_df), len(jobs_df)
    F_raw = np.zeros((N_seek, N_job, 8), dtype=np.float32)
    all_skills = [skill for skills in seekers_df['technical_skills'] for skill in skills.split(', ') if skill]
    counts = pd.Series(all_skills).value_counts()
    tot_docs = N_seek + N_job
    skill_idf = {s: np.log((tot_docs+1)/(cnt+1))+1 for s, cnt in counts.items()}
    all_sals = np.concatenate([seekers_df['salary'], jobs_df['salary_offer']])
    max_sal, min_sal = np.percentile(all_sals, 95), np.percentile(all_sals,5)
    max_edu = seekers_df['edu_value'].max()
    dist = np.full((N_seek, N_job), np.nan)
    for i in range(N_seek):
        for j in range(N_job):
            c1, c2 = seekers_coords[i], jobs_coords[j]
            if not math.isnan(c1[0]) and not math.isnan(c2[0]):
                dist[i,j] = haversine(c1, c2)
    max_dist = np.nanmax(dist) if not np.all(np.isnan(dist)) else 1.0
    for i in range(N_seek):
        for j in range(N_job):
            common = seeker_skills[i] & job_skills[j]
            missing = job_skills[j] - seeker_skills[i]
            penalty = 1 - len(missing)/len(job_skills[j]) if job_skills[j] else 0
            ms = sum(skill_idf.get(s,0) for s in common)
            ts = sum(skill_idf.get(s,0) for s in job_skills[j]) if job_skills[j] else 0
            F_raw[i,j,0] = penalty * (ms/ts if ts > 0 else 0)
            F_raw[i,j,1] = min(seekers_df.at[i,'years_experience']/max(jobs_df.at[j,'min_experience'],1), 1.5)
            diff = abs(jobs_df.at[j,'salary_offer'] - seekers_df.at[i,'salary'])
            F_raw[i,j,2] = 1 - np.log1p(diff)/np.log1p(max_sal - min_sal)
            F_raw[i,j,3] = seekers_df.at[i,'edu_value'] / max_edu
            F_raw[i,j,4] = (seek_sector[i] == job_sector[j])
            F_raw[i,j,5] = (seek_contract[i] == job_contract[j])
            F_raw[i,j,6] = seekers_df.at[i,'edu_value'] / 20
            d = dist[i,j] if not np.isnan(dist[i,j]) else max_dist
            F_raw[i,j,7] = 1 - (d / (max_dist + 1e-8))
    mins = F_raw.min(axis=(0,1))
    maxs = F_raw.max(axis=(0,1))
    F = (F_raw - mins) / (maxs - mins + 1e-8)
    return F

class JobMatchingProblem:
    def __init__(self, F, valid_jobs, weights):
        self.F = F
        self.valid_jobs = valid_jobs
        self.weights = weights
        self.num_seekers = F.shape[0]
        self.num_jobs = F.shape[1]
    
    def get_size(self):
        return self.num_seekers
    
    def generate_individual(self):
        individual = []
        for i in range(self.num_seekers):
            valid = np.append(np.where(self.valid_jobs[i])[0], -1)
            individual.append(np.random.choice(valid))
        return individual
    
    def evaluate(self, chromosome):
        total_score = 0.0
        for i, job_idx in enumerate(chromosome):
            if job_idx == -1:
                total_score -= 0.1
            else:
                if self.valid_jobs[i, job_idx]:
                    total_score += np.dot(self.F[i, job_idx], self.weights)
                else:
                    total_score -= 1.0
        return -total_score  # Minimize this value

class GeneticAlgorithm:
    def __init__(self, problem, population_size=100, generations=500, mutation_rate=0.01,
                 elite_size=2, tournament_size=3, selection_method='tournament',
                 crossover_method='pmx', mutation_method='swap'):
        self.problem = problem
        self.population_size = population_size
        self.generations = generations
        self.mutation_rate = mutation_rate
        self.elite_size = elite_size
        self.tournament_size = tournament_size
        
        if selection_method == 'tournament':
            self.selection_fn = self.tournament_selection
        elif selection_method == 'roulette':
            self.selection_fn = self.roulette_wheel_selection
        else:
            raise ValueError(f"Unknown selection method: {selection_method}")
        
        if crossover_method == 'single_point':
            self.crossover_fn = self.single_point_crossover
        elif crossover_method == 'pmx':
            self.crossover_fn = self.pmx_crossover
        elif crossover_method == 'order':
            self.crossover_fn = self.order_crossover
        else:
            raise ValueError(f"Unknown crossover method: {crossover_method}")
        
        if mutation_method == 'custom':
            self.mutation_fn = self.custom_mutation
        elif mutation_method == 'swap':
            self.mutation_fn = self.swap_mutation
        else:
            raise ValueError(f"Unknown mutation method: {mutation_method}")
    
    def tournament_selection(self, population, fitnesses):
        contenders = random.sample(range(len(population)), self.tournament_size)
        best_idx = min(contenders, key=lambda i: fitnesses[i])
        return population[best_idx]
    
    def roulette_wheel_selection(self, population, fitnesses):
        max_fitness = max(fitnesses)
        transformed = [max_fitness - f + 1e-10 for f in fitnesses]
        total = sum(transformed)
        if total == 0:
            return random.choice(population)
        point = random.uniform(0, total)
        current = 0
        for i, t in enumerate(transformed):
            current += t
            if current >= point:
                return population[i]
        return population[-1]
    
    def single_point_crossover(self, parent1, parent2):
        size = len(parent1)
        if size <= 1:
            return parent1.copy(), parent2.copy()
        cut = random.randint(1, size-1)
        return parent1[:cut] + parent2[cut:], parent2[:cut] + parent1[cut:]
    
    def pmx_crossover(self, parent1, parent2):
        size = len(parent1)
        cx1, cx2 = sorted(random.sample(range(size), 2))
        child1, child2 = [-1]*size, [-1]*size
        for i in range(cx1, cx2+1):
            child1[i], child2[i] = parent1[i], parent2[i]
        mapping1 = {parent1[i]: parent2[i] for i in range(cx1, cx2+1)}
        mapping2 = {parent2[i]: parent1[i] for i in range(cx1, cx2+1)}
        # Fill remaining positions (omitted for brevity)
        return child1, child2
    
    def order_crossover(self, parent1, parent2):
        size = len(parent1)
        cx1, cx2 = sorted(random.sample(range(size), 2))
        child1, child2 = [-1]*size, [-1]*size
        for i in range(cx1, cx2+1):
            child1[i], child2[i] = parent1[i], parent2[i]
        # Fill remaining positions (omitted for brevity)
        return child1, child2
    
    def swap_mutation(self, individual, mutation_rate=None):
        rate = mutation_rate or self.mutation_rate
        if random.random() < rate:
            pos1, pos2 = random.sample(range(len(individual)), 2)
            individual[pos1], individual[pos2] = individual[pos2], individual[pos1]
        return individual
    
    def custom_mutation(self, individual, mutation_rate=None):
        rate = mutation_rate or self.mutation_rate
        mutated = individual.copy()
        for i in range(len(mutated)):
            if random.random() < rate:
                valid = np.append(np.where(self.problem.valid_jobs[i])[0], -1)
                mutated[i] = random.choice(valid)
        return mutated
    
    def evolve_population(self, population, fitnesses):
        new_pop = []
        sorted_indices = sorted(range(len(fitnesses)), key=lambda i: fitnesses[i])
        for i in range(self.elite_size):
            new_pop.append(population[sorted_indices[i]])
        while len(new_pop) < self.population_size:
            parent1 = self.selection_fn(population, fitnesses)
            parent2 = self.selection_fn(population, fitnesses)
            child1, child2 = self.crossover_fn(parent1, parent2)
            new_pop.append(self.mutation_fn(child1))
            if len(new_pop) < self.population_size:
                new_pop.append(self.mutation_fn(child2))
        return new_pop
    
    def solve(self):
        population = [self.problem.generate_individual() for _ in range(self.population_size)]
        best_individual = None
        best_fitness = float('inf')
        for gen in range(self.generations):
            fitnesses = [self.problem.evaluate(ind) for ind in population]
            current_best = min(fitnesses)
            if current_best < best_fitness:
                best_fitness = current_best
                best_individual = population[fitnesses.index(current_best)].copy()
                print(f"Generation {gen}: Best Score {-best_fitness:.2f}")
            population = self.evolve_population(population, fitnesses)
        return best_individual, -best_fitness

if __name__ == '__main__':
    seekers_df = load_data()
    jobs_df = input_job_details(seekers_df)
    args = preprocess_data(seekers_df, jobs_df)
    F = calculate_features(*args)
    WEIGHTS = np.array([0.40,0.15,0.15,0.10,0.10,0.05,0.05,0.10])
    valid = (seekers_df['edu_rank'].values[:, None] >= jobs_df['edu_rank'].values) & \
            (seekers_df['years_experience'].values[:, None] >= jobs_df['min_experience'].values) & \
            np.array([[len(s & j) >= max(1, len(j)//2) for j in args[3]] for s in args[2]])
    problem = JobMatchingProblem(F, valid, WEIGHTS)
    ga = GeneticAlgorithm(problem, population_size=50, generations=100, mutation_rate=0.1,
                          elite_size=2, tournament_size=3, selection_method='tournament',
                          crossover_method='single_point', mutation_method='custom')
    best_solution, best_score = ga.solve()
    print("\n=== Matching Results ===")
    print(f"Best Score: {best_score:.2f}\n")
    print("\n=== Top 5 Candidates per Job ===")
    for j in range(len(jobs_df)):
        scores = F[:,j].dot(WEIGHTS)
        valid_idx = np.where(valid[:,j])[0]
        ranked = sorted([(i, scores[i]) for i in valid_idx], key=lambda x: -x[1])[:5]
        print(f"\nJob {jobs_df.at[j,'job_id']}:")
        for rank, (i, score) in enumerate(ranked, 1):
            print(f"  {rank}. {seekers_df.at[i,'employee_Id']} (Score: {score:.2%})")


=== Enter Job Details ===
Generation 0: Best Score -767.78
Generation 1: Best Score -766.11
Generation 2: Best Score -762.90
Generation 3: Best Score -759.44
Generation 5: Best Score -759.18
Generation 6: Best Score -757.72
Generation 7: Best Score -756.28
Generation 10: Best Score -753.66
Generation 13: Best Score -752.95
Generation 19: Best Score -751.94
Generation 22: Best Score -751.47
Generation 23: Best Score -749.60
Generation 24: Best Score -749.38
Generation 29: Best Score -747.77
Generation 38: Best Score -746.80
Generation 44: Best Score -746.33
Generation 45: Best Score -745.40
